# 实验5 · 基于视觉的目标识别（Ch5 目标识别）

**课程**：Ch5 目标识别　　**对应手册**：`7.1 使用 OpenCV 进行颜色检测`（BGR/HSV、颜色识别与追踪）、`7.13.1 颜色校准`

**课堂任务**：在**固定 ROI** 内识别红、黄、绿、蓝方块，并在画面上显示类别（或输出类别字符串）。

**提交成果**

| 成果 | 文件 |
| --- | --- |
| 验收 notebook | 本文件 |
| 四种颜色 HSV 参数表 | `hsv_params_table.csv`、`hsv_config.json` |
| 40 次识别测试记录 | `test_record.csv` |

**本 notebook 和已有源码的关系（不修改任何已有源码）**

| 复用的已有文件 | 作用 |
| --- | --- |
| `camera.py`（本目录） | 平台相机封装：读 `vconfig/camera.yaml`、处理镜像/ROI。**可选**，没硬件时自动降级 |
| `data/roi_image/{red,yellow,green,blue}/*.png` | 平台自带的真实色块样本，用于离线演示与 HSV 标定 |
| `1.使用OpenCV进行颜色检测.ipynb` | `cv2.cvtColor` → `cv2.inRange` → Mask 的写法来源 |
| `color_recognition.py` / `week5_color_recognition.ipynb` | 同题目的现成实现，可直接 `import` 与本文对比 |
| `13.OpenCV 摄像头色块抓取分拣实验.ipynb` + `color_block_detector.py` + `grab_color_block.py` | 色块→机械臂抓取链路（本实验的下一阶段） |

> **不需要**运行相机内参标定 / 手眼九点标定（`11.摄像头Radon标定.ipynb`、`12.OpenCV 手眼九点标定工作台.ipynb`、`camera_calibration.py`）。
> 本实验说的“标定”是 **HSV 阈值标定**（见 §4），和相机标定不是一回事。只有继续做“识别→抓取”（把像素坐标换算成工作台坐标）时才用得到那些标定文件。

**运行方法**：把本 notebook 上传到 Jupyter 里，从上到下依次运行（Shift+Enter）。
**不用改任何路径**：notebook 会自己去找 `camera.py` / `vconfig` / `data/roi_image` / `simhei.ttf`。
自动选择相机：平台相机 `camera.py` → 普通 USB 摄像头 `cv2.VideoCapture(0)` → 内置演示画面 + 真实样本，保证一定能跑通。

---
## §0　环境准备

只做 import 和工具函数，不产生识别结果。

**路径说明**：本 notebook **不写死任何路径**。它会在【当前目录 → notebook 所在目录 → 上 1~3 级目录 → 环境变量 `ROBOT_CV_DIRS`】里
自动查找 `camera.py`、`vconfig`、`data/roi_image`、`simhei.ttf`；**一处都找不到也不会报错**，会自动降级为
`cv2.VideoCapture` + 内置合成画面 + 英文标签。结果文件默认写回 notebook 所在目录。

In [ ]:
# ============================ 环境准备 ============================
import os, sys, json, csv, time, glob, ctypes, inspect, threading, tempfile
from datetime import datetime

import cv2
import numpy as np

try:
    import ipywidgets as widgets
    from IPython.display import display, clear_output
    HAS_WIDGETS = True
except Exception as _e:
    widgets = None
    HAS_WIDGETS = False
    print('[WARN] ipywidgets 不可用:', _e)


# ---------------------------------------------------------------------------
# 依赖文件自动定位：本 notebook 不写死任何路径
# 查找顺序：当前目录 -> notebook 所在目录 -> 上 1~3 级目录 -> 环境变量 ROBOT_CV_DIRS
# 查找目标：camera.py、vconfig、data/roi_image、simhei.ttf
# 全都找不到也不会报错，后面会自动降级（cv2.VideoCapture + 内置合成画面）
# ---------------------------------------------------------------------------
def _candidate_dirs():
    cands = [os.path.abspath(os.getcwd())]
    for key in ('__vsc_ipynb_file__', '__session__', '__notebook__'):
        v = globals().get(key)
        if isinstance(v, str):
            cands.append(os.path.dirname(os.path.abspath(v)))
    for d in list(cands):
        p = d
        for _ in range(3):
            p = os.path.dirname(p)
            if p:
                cands.append(p)
    for d in os.environ.get('ROBOT_CV_DIRS', '').split(os.pathsep):
        if d.strip():
            cands.append(os.path.abspath(d.strip()))
    seen, out = set(), []
    for d in cands:
        if d and d not in seen and os.path.isdir(d):
            seen.add(d)
            out.append(d)
    return out


def find_dep(name, kind='any', depth=2):
    """先在各候选目录直接命中，再向下做浅层（默认 2 级）递归搜索。"""
    for d in _candidate_dirs():
        p = os.path.join(d, name)
        if (kind == 'file' and os.path.isfile(p)) or \
           (kind == 'dir' and os.path.isdir(p)) or \
           (kind == 'any' and os.path.exists(p)):
            return p
    base = os.path.basename(name)
    for d in _candidate_dirs():
        base_depth = d.rstrip('/\\').count(os.sep)
        for root, dirs, files in os.walk(d):
            if root.count(os.sep) - base_depth >= depth:
                dirs[:] = []
                continue
            if base in (dirs if kind == 'dir' else files):
                return os.path.join(root, base)
    return None


CAMERA_PY   = find_dep('camera.py', 'file')
COURSE_DIR  = os.path.dirname(CAMERA_PY) if CAMERA_PY else os.path.abspath(os.getcwd())
if COURSE_DIR not in sys.path:
    sys.path.insert(0, COURSE_DIR)      # 保证 import camera / arm5dof_uservo 找得到

VCONFIG_DIR = find_dep('vconfig', 'dir')
DATA_DIR    = find_dep(os.path.join('data', 'roi_image'), 'dir')
FONT_PATH   = find_dep('simhei.ttf', 'file')

OUTDIR = os.path.abspath(os.getcwd())   # 结果写回 notebook 所在目录
try:
    _probe = os.path.join(OUTDIR, '.write_test')
    open(_probe, 'w').close(); os.remove(_probe)
except Exception:
    OUTDIR = tempfile.gettempdir()      # 只读环境退回临时目录

print('Python       :', sys.version.split()[0])
print('OpenCV       :', cv2.__version__)
print('ipywidgets   :', HAS_WIDGETS)
print('依赖目录     :', COURSE_DIR)
print('camera.py    :', CAMERA_PY or '未找到 -> 将直接用 cv2.VideoCapture(0)')
print('vconfig      :', VCONFIG_DIR or '未找到')
print('样本目录     :', DATA_DIR or '未找到 -> 使用合成样本')
print('中文字体     :', FONT_PATH or '未找到 -> 画面标签自动改用英文')
print('结果输出目录 :', OUTDIR)
if not CAMERA_PY:
    print('提示：把本 notebook 和实验4 的 camera.py / data / simhei.ttf / vconfig 放同一目录即可；')
    print('      也可指定环境变量后重跑本 cell：os.environ["ROBOT_CV_DIRS"] = "<实验4目录的绝对路径>"')


def imread_bgr(path):
    """读图片。用 imdecode + fromfile，兼容 Windows 上的中文/非 ASCII 路径（cv2.imread 会读不到）。"""
    try:
        data = np.fromfile(path, dtype=np.uint8)
        img = cv2.imdecode(data, cv2.IMREAD_COLOR)
        if img is not None:
            return img
    except Exception:
        pass
    return cv2.imread(path)


def bgr8_to_jpeg(bgr, quality=75):
    """BGR 图像 -> JPEG 字节流（与实验4 其它 notebook 的显示方式一致）。"""
    return bytes(cv2.imencode('.jpg', bgr, [int(cv2.IMWRITE_JPEG_QUALITY), quality])[1])


def show(bgr, width=640):
    """在 notebook 里显示一张 BGR 图像。"""
    if bgr.ndim == 2:
        bgr = cv2.cvtColor(bgr, cv2.COLOR_GRAY2BGR)
    if HAS_WIDGETS:
        w = widgets.Image(format='jpeg', width=width)
        w.value = bgr8_to_jpeg(bgr)
        display(w)
        return w
    from IPython.display import Image as _Img, display as _d
    return _d(_Img(data=bgr8_to_jpeg(bgr)))


# 中文字体：用自动找到的 simhei.ttf；找不到就自动改用英文标签（OpenCV 自带字体不支持中文）
try:
    from PIL import Image, ImageDraw, ImageFont
    _HAS_FONT = bool(FONT_PATH) and os.path.exists(FONT_PATH)
except Exception:
    _HAS_FONT = False


def put_text_cn(img_bgr, cn_text, en_text, org, color=(255, 255, 255), size=20):
    """在图上写中文；没有字体或没有 PIL 时退化成 OpenCV 英文写法。"""
    if _HAS_FONT:
        try:
            font = ImageFont.truetype(FONT_PATH, size)
            pil = Image.fromarray(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB))
            ImageDraw.Draw(pil).text(org, cn_text, font=font, fill=(color[2], color[1], color[0]))
            return cv2.cvtColor(np.array(pil), cv2.COLOR_RGB2BGR)
        except Exception:
            pass
    cv2.putText(img_bgr, en_text, org, cv2.FONT_HERSHEY_SIMPLEX, 0.8, color, 2, cv2.LINE_AA)
    return img_bgr


# 强制结束线程（和实验4 其它 notebook 一致，用来收掉实时循环）
def _async_raise(tid, exctype):
    tid = ctypes.c_long(tid)
    if not inspect.isclass(exctype):
        exctype = type(exctype)
    res = ctypes.pythonapi.PyThreadState_SetAsyncExc(tid, ctypes.py_object(exctype))
    if res == 0:
        raise ValueError('invalid thread id')
    elif res != 1:
        ctypes.pythonapi.PyThreadState_SetAsyncExc(tid, None)
        raise SystemError('PyThreadState_SetAsyncExc failed')


def stop_thread(thread):
    try:
        _async_raise(thread.ident, SystemExit)
    except Exception as e:
        print('stop_thread:', e)
# ---------------------------------------------------------------------------
# 实时预览 / 实时识别（ipywidgets 显示，后台线程持续刷新画面）
#   start_live(annotate=False) -> 只看实时原图
#   start_live(recognizer=reco) -> 实时叠加 ROI 框 / 色块框 / 识别标签
# ---------------------------------------------------------------------------
_LIVE = {'thread': None, 'state': None}


def stop_live(quiet=False):
    """停止当前的实时循环：先温和退出，1 秒还退不掉再强制结束线程。"""
    st, th = _LIVE.get('state'), _LIVE.get('thread')
    if st is not None:
        st['stop'] = True
    if th is not None:
        try:
            th.join(timeout=1.0)
        except Exception:
            pass
        if th.is_alive():
            try:
                stop_thread(th)
            except Exception:
                pass
        if not quiet:
            print('已停止实时循环。')
    _LIVE['thread'] = None
    _LIVE['state'] = None


def start_live(recognizer=None, cam=None, width=640, height=480, fps=20,
               annotate=True, auto_start=True):
    """实时显示摄像头画面。

    同一时刻只保留一个循环：本 cell 重复运行会自动停掉上一个，不会两个线程抢摄像头。
    """
    if not auto_start:
        print('auto_start=False，未启动。改成 True 或手动调用 start_live()。')
        return None
    if not HAS_WIDGETS:
        print('没有 ipywidgets，无法实时显示；请用单帧识别单元格演示。')
        return None
    stop_live(quiet=True)
    cam = cam or globals().get('camera')
    img_w = widgets.Image(format='jpeg', width=width)
    info = widgets.HTML()
    btn = widgets.Button(description='停止实时', button_style='danger')
    display(widgets.VBox([img_w, widgets.HBox([btn, info])]))
    state = {'stop': False, 'fps': 0.0}
    _LIVE['state'] = state

    def loop():
        t0 = time.time()
        while not state['stop']:
            try:
                ok, f = cam.read()
            except Exception:
                break
            if not ok or f is None:
                time.sleep(0.05)
                continue
            f = cv2.resize(f, (width, height))
            if annotate and recognizer is not None:
                lab, _blk, det = recognizer.classify(f)
                f = recognizer.annotate(f, det, label=lab)
            img_w.value = bgr8_to_jpeg(f)
            now = time.time()
            state['fps'] = 0.9 * state['fps'] + 0.1 / max(1e-3, now - t0)
            t0 = now
            info.value = '<b>实时中… %.1f FPS</b>' % state['fps']
            time.sleep(1.0 / fps)

    def on_stop(_b):
        state['stop'] = True
        btn.description = '已停止'
        btn.disabled = True

    btn.on_click(on_stop)
    th = threading.Thread(target=loop, daemon=True)
    _LIVE['thread'] = th
    th.start()
    print('实时循环已启动。结束方式：点上面的「停止实时」按钮，或运行 stop_live()。')
    return th


In [ ]:
# ---------------- 颜色常量（OpenCV 的 HSV：H∈[0,180], S∈[0,255], V∈[0,255]） ----------------
COLOR_ORDER = ['red', 'yellow', 'green', 'blue']
COLOR_CN    = {'red': '红色', 'yellow': '黄色', 'green': '绿色', 'blue': '蓝色', 'none': '未识别'}
COLOR_BGR   = {'red': (0, 0, 255), 'yellow': (0, 255, 255), 'green': (0, 255, 0), 'blue': (255, 0, 0)}

# 初始 HSV 阈值（手册 7.1 的经验值；红色 H 跨 0 度，代码里会自动补第二段）
DEFAULT_HSV = {
    'red':    ((0,   70, 50), (10,  255, 255)),
    'yellow': ((26,  70, 50), (34,  255, 255)),
    'green':  ((35,  70, 50), (77,  255, 255)),
    'blue':   ((100, 70, 50), (124, 255, 255)),
}

# 课堂要求的固定 ROI（x, y, w, h）：方块必须放进这个框里
ROI = (160, 100, 320, 300)


def make_demo_frame(blocks=None, bg=235, size=(480, 640), jitter=0):
    """合成一张“工作台”：浅色背景 + 彩色方块。
    没接摄像头时用它兜底，保证整份 notebook 一定能演示。"""
    h, w = size
    img = np.full((h, w, 3), bg, np.uint8)
    # 只把“红色”方块放进固定 ROI(160,100,320,300)，另外三色放 ROI 外，
    # 用来演示“ROI 外的方块不参与识别”
    blocks = blocks or [('red',    (210, 190, 120, 110)),
                        ('yellow', (20,  60,  100, 90)),
                        ('green',  (520, 320, 100, 90)),
                        ('blue',   (30,  330, 100, 90))]
    for color, (x, y, bw, bh) in blocks:
        cv2.rectangle(img, (x, y), (x + bw, y + bh), COLOR_BGR[color], -1)
        cv2.rectangle(img, (x, y), (x + bw, y + bh), (60, 60, 60), 2)
    if jitter:
        img = np.clip(img.astype(np.int16) + np.random.randint(-jitter, jitter + 1, img.shape), 0, 255).astype(np.uint8)
    return cv2.GaussianBlur(img, (3, 3), 0)


DEMO_FRAME = make_demo_frame()
x, y, w, h = ROI
cv2.rectangle(DEMO_FRAME, (x, y), (x + w, y + h), (255, 200, 0), 3)
show(DEMO_FRAME)

---
## §0.5　上机准备：把机械臂摆到「拍摄位」

实验箱上的**相机装在机械臂末端**，只有机械臂保持在 `capture_image`（拍摄工作台）位姿时，相机才正对工作台。
所以**接上实验箱后、运行 §1 之前，先跑下面这一个 cell**（它只做摆位，和相机内参标定无关）。

- 没接机械臂 / 在电脑上预习：可以直接跳过，notebook 会自动使用演示画面。
- 等价终端命令：`python set_arm_pose_by_name.py --pose_name=capture_image`


In [ ]:
# ---------- 上机第一步：机械臂摆到拍摄位 capture_image ----------
# 说明：相机装在机械臂末端，只有摆到拍摄位，§1 读到的画面才是工作台。
MOVE_ARM_TO_CAPTURE_POSE = True     # 没接机械臂时留着也不影响（异常会被忽略）

if MOVE_ARM_TO_CAPTURE_POSE:
    try:
        from arm5dof_uservo import Arm5DoFUServo
        arm = Arm5DoFUServo()                                # 打开串口并使能舵机
        arm.set_tool_pose(pose_name='capture_image')         # 位姿定义在 config.py 的 ARM_POSE_DICT
        print('机械臂已移动到 capture_image 拍摄位姿，可以开始识别了。')
    except Exception as e:
        arm = None
        print('[INFO] 没连接机械臂（%s），跳过摆位。' % repr(e)[:70])
        print('       在实验箱上请确认：已上电、串口号正确（config.py 的 DEVICE_PORT_DEFAULT）、机械臂没有被挡住。')
else:
    print('已跳过摆位（MOVE_ARM_TO_CAPTURE_POSE = False）。')


---
## §1　第 1 学时：摄像头与图像采集

读 Frame、理解 **BGR** 通道顺序，并在画面里画出固定 ROI。

本节包含：① 打开相机 → ② 抓**一帧**讲 BGR（静止截图）→ ③ **实时预览**（连续画面）。

In [ ]:
# ---------- 统一相机接口：平台 camera.py → cv2.VideoCapture → 演示画面 ----------
class DemoCamera(object):
    """兜底相机：永远返回合成画面（没硬件也能演示）。"""
    def read(self):
        return True, make_demo_frame(jitter=4)
    def release(self):
        pass


def open_camera(index=0, width=640, height=480):
    # 1) 优先复用实验4 的平台相机封装 camera.py
    #    vconfig 目录用自动找到的路径，不依赖当前工作目录
    try:
        from camera import Camera
        try:
            cam = Camera(id=index, config_folder=VCONFIG_DIR) if VCONFIG_DIR else Camera(id=index)
        except TypeError:
            cam = Camera(id=index)
        ok, f = cam.read()
        if ok and f is not None:
            for _ in range(5):      # 丢掉前几帧：UVC 摄像头开头常是黑屏/曝光未稳
                cam.read()
            print('已打开【平台 camera.py】相机, frame.shape =', f.shape)
            return cam
        print('[WARN] 平台相机没读到画面，改用 cv2.VideoCapture ...')
    except Exception as e:
        print('[INFO] 平台 camera.py 不可用：', repr(e)[:80])
    # 2) OpenCV 直接打开（0 失败时再试 1，Jetson 上常见 video1）
    for idx in ([index, 1] if index == 0 else [index]):
        try:
            cap = cv2.VideoCapture(idx)
            cap.set(cv2.CAP_PROP_FRAME_WIDTH, width)
            cap.set(cv2.CAP_PROP_FRAME_HEIGHT, height)
            ok, f = cap.read()
            if ok and f is not None:
                for _ in range(5):  # 丢掉前几帧：UVC 摄像头开头常是黑屏/曝光未稳
                    cap.read()
                print('已打开【cv2.VideoCapture(%d)】, frame.shape =' % idx, f.shape)
                return cap
            cap.release()
        except Exception as e:
            print('[INFO] cv2.VideoCapture(%d) 失败：%s' % (idx, repr(e)[:60]))
    # 3) 兜底
    print('未检测到摄像头 -> 使用内置演示画面（后面的识别流程完全一致）。')
    return DemoCamera()


camera = open_camera(0)

In [ ]:
# ---------- 读取一帧：理解 BGR ----------
ok, frame = camera.read()
print('read() ->', ok, '，frame.shape =', None if frame is None else frame.shape)   # (高, 宽, 3)

frame = cv2.resize(frame, (640, 480))     # 统一成 640x480，ROI 才有固定含义
b, g, r = cv2.split(frame)                # OpenCV 读进来是 BGR：0=B  1=G  2=R
print('通道顺序：0 = B(蓝)   1 = G(绿)   2 = R(红)')
px = tuple(int(v) for v in frame[240, 320])
print('画面中心像素 BGR = %s  ->  (B=%d, G=%d, R=%d)' % (px, px[0], px[1], px[2]))
print('注意：如果直接把这个数组当 RGB 用（送进 PIL/matplotlib），红蓝会反过来 —— 这就是必须先讲 BGR 的原因。')

x, y, w, h = ROI
vis = frame.copy()
cv2.rectangle(vis, (x, y), (x + w, y + h), (255, 200, 0), 3)
vis = put_text_cn(vis, '固定识别区 ROI', 'ROI', (x + 8, y + 8), (255, 200, 0), 24)
show(np.hstack([vis,
                cv2.cvtColor(b, cv2.COLOR_GRAY2BGR),
                cv2.cvtColor(g, cv2.COLOR_GRAY2BGR),
                cv2.cvtColor(r, cv2.COLOR_GRAY2BGR)]))
print('左起：原图(BGR) | B通道 | G通道 | R通道')

---
### §1.2　实时画面预览

`camera.read()` 只抓一帧，所以上面看到的是**静止截图**；下面这个 cell 才是**实时**画面（后台线程持续刷新 + 画面下方显示实时帧率）。
点画面上的「停止实时」按钮即可结束。

In [ ]:
# ---------- §1.2 实时画面预览（要看「实时」，跑的是这一段） ----------
# 上面那个 cell 里 camera.read() 只抓一帧，是故意用来讲 BGR 通道的；
# 想一直看到连续画面，跑本 cell。画面下方会显示实时帧率。
t_live = start_live(annotate=False)

In [ ]:
# ---------- 离线样本：平台自带的真实色块图 data/roi_image ----------
def load_sample_images(folder=None):
    """读取平台真实色块样本 {颜色: 图片}；目录找不到就返回空（后面用合成图兜底）。"""
    folder = folder or DATA_DIR
    out = {}
    if not folder or not os.path.isdir(folder):
        return out
    for color in COLOR_ORDER:
        files = sorted(glob.glob(os.path.join(folder, color, '*.png')))
        if files:
            img = imread_bgr(files[0])
            if img is not None:
                out[color] = cv2.resize(img, (160, 200))
    return out


SAMPLES = load_sample_images()
print('样本目录:', DATA_DIR)
print('真实样本：', {COLOR_CN[k]: v.shape for k, v in SAMPLES.items()} or '无（不影响后续运行）')
if SAMPLES:
    show(np.hstack([SAMPLES[c] for c in COLOR_ORDER if c in SAMPLES]))

---
## §2　第 2 学时：BGR→HSV 与 Mask

`cv2.cvtColor` 做颜色空间转换，`cv2.inRange` 做二值化得到 Mask —— 这是整个颜色识别的核心两步。

- **BGR**：每个像素是蓝/绿/红三通道亮度，受光照强弱影响很大。
- **HSV**：H 色调（颜色种类）、S 饱和度（颜色的浓淡）、V 明度（亮暗）。
- 识别**颜色种类**主要看 H，所以光照变化对 HSV 的干扰比 BGR 小得多。

In [ ]:
# ---------- BGR -> HSV，并用 cv2.inRange 生成 Mask ----------
img = DEMO_FRAME.copy()
hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)          # ← 第 1 步：颜色空间转换

lower = np.array(DEFAULT_HSV['red'][0], np.uint8)
upper = np.array(DEFAULT_HSV['red'][1], np.uint8)
mask = cv2.inRange(hsv, lower, upper)               # ← 第 2 步：阈值分割
# 红色在 H 上跨 0 度，需要补第二段（170~180 和 0~10 其实是同一片红）
mask |= cv2.inRange(hsv, np.array([180 - upper[0], lower[1], lower[2]], np.uint8),
                    np.array([180 - lower[0], upper[1], upper[2]], np.uint8))

print('lower =', lower.tolist(), ' upper =', upper.tolist())
print('inRange 之后 Mask 只有两种取值：', np.unique(mask))
print('白色像素占比 %.2f%%' % (float((mask > 0).mean()) * 100))

vis = img.copy()
cnts = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)[-2]
cv2.drawContours(vis, cnts, -1, (0, 255, 255), 3)
show(np.hstack([vis, cv2.cvtColor(mask, cv2.COLOR_GRAY2BGR)]))
print('左：原图 + 轮廓    右：二值 Mask')

In [ ]:
# ---------- 颜色识别的通用工具函数（复用实验4“1.使用OpenCV进行颜色检测”的写法） ----------
def to_hsv(img_bgr):
    """BGR -> HSV。OpenCV 读进来就是 BGR，不是 RGB。"""
    return cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)


def clean_mask(mask, k_open=3, k_close=5):
    """开运算去掉小噪点，闭运算补上反光造成的空洞。"""
    if k_open > 0:
        ko = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (k_open, k_open))
        mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, ko)
    if k_close > 0:
        kc = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (k_close, k_close))
        mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kc)
    return mask


def color_ranges(color, params):
    """返回某颜色需要 inRange 的若干段 (lower, upper)。红色跨 0 度，要两段。"""
    lower, upper = params[color]
    lower = tuple(int(v) for v in lower)
    upper = tuple(int(v) for v in upper)
    ranges = [(lower, upper)]
    if color == 'red':
        lo2, up2 = (180 - upper[0], lower[1], lower[2]), (180 - lower[0], upper[1], upper[2])
        if lo2[0] < up2[0]:
            ranges.append((lo2, up2))
    return ranges


def build_mask(hsv, color, params, clean=True):
    """用 cv2.inRange 生成二值 Mask（红色两段取或）。"""
    mask = np.zeros(hsv.shape[:2], np.uint8)
    for lower, upper in color_ranges(color, params):
        mask |= cv2.inRange(hsv, np.array(lower, np.uint8), np.array(upper, np.uint8))
    return clean_mask(mask) if clean else mask


def find_blocks(mask, min_area=800, max_area=200000, min_fill=0.55, max_wh_ratio=2.0, offset=(0, 0)):
    """在 Mask 里找方形色块：面积范围 + 长宽比接近 1 + 填充率足够大 三重过滤。"""
    contours = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)[-2]   # 兼容 OpenCV 3/4/5
    ox, oy = offset
    blocks = []
    for cnt in contours:
        area = cv2.contourArea(cnt)
        if area < min_area or area > max_area:
            continue
        x, y, w, h = cv2.boundingRect(cnt)
        if w <= 0 or h <= 0 or max(w, h) / float(min(w, h)) > max_wh_ratio:
            continue
        if area / float(w * h) < min_fill:
            continue
        blocks.append({'rect': (x + ox, y + oy, w, h),
                       'center': (x + ox + w // 2, y + oy + h // 2),
                       'area': area})
    blocks.sort(key=lambda b: b['area'], reverse=True)
    return blocks


def mean_hsv(img_bgr, roi=None):
    """求（ROI 内）HSV 均值，用于 §4 标定。"""
    patch = img_bgr if roi is None else img_bgr[roi[1]:roi[1] + roi[3], roi[0]:roi[0] + roi[2]]
    if patch is None or patch.size == 0:
        return None
    v = to_hsv(patch).reshape(-1, 3).astype(np.float32)
    return tuple(round(float(t), 1) for t in v.mean(axis=0))


print('工具函数已就绪：to_hsv / clean_mask / color_ranges / build_mask / find_blocks / mean_hsv')

In [ ]:
# ---------- 四种颜色各生成一张 Mask，直观看出阈值分离效果 ----------
HSV_PARAMS = {k: (tuple(v[0]), tuple(v[1])) for k, v in DEFAULT_HSV.items()}
hsv = to_hsv(DEMO_FRAME)
preview = [DEMO_FRAME]
for color in COLOR_ORDER:
    m = build_mask(hsv, color, HSV_PARAMS)
    preview.append(cv2.cvtColor(m, cv2.COLOR_GRAY2BGR))
show(np.hstack(preview))
print('左起：原图 | 红Mask | 黄Mask | 绿Mask | 蓝Mask')
print('每一步都只对 ROI 区域做也行（见 §3），整图做只是方便看效果。')

---
## §3　第 3 学时：四色识别（固定 ROI + 轮廓 + 类别输出）

流程：**ROI 裁剪 → BGR→HSV → 四色 inRange → 形态学去噪 → 找轮廓 → 面积/形状过滤 → 面积最大的色块即为识别结果**。

In [ ]:
# ---------- 四色识别器：在固定 ROI 内输出颜色类别 ----------
class ColorRecognizer(object):
    def __init__(self, params=None, roi=ROI):
        self.params = {k: (tuple(v[0]), tuple(v[1])) for k, v in (params or DEFAULT_HSV).items()}
        self.roi = roi

    def detect(self, img_bgr, roi=None):
        """返回 {颜色: [色块, ...]}，坐标已换算回原图。"""
        roi = roi or self.roi
        x, y, w, h = roi
        hsv = to_hsv(img_bgr[y:y + h, x:x + w])        # 只处理 ROI，能显著减少背景干扰
        out = {}
        for color in COLOR_ORDER:
            out[color] = find_blocks(build_mask(hsv, color, self.params), offset=(x, y))
        return out

    def classify(self, img_bgr, roi=None):
        """ROI 内取面积最大的色块，返回 (颜色类别, 色块, 全部检测结果)。"""
        det = self.detect(img_bgr, roi)
        best_color, best_block, best_area = 'none', None, 0
        for color, blocks in det.items():
            if blocks and blocks[0]['area'] > best_area:
                best_color, best_block, best_area = color, blocks[0], blocks[0]['area']
        return best_color, best_block, det

    def annotate(self, img_bgr, det, label=None, roi=None):
        """画出 ROI 框、各色色块框和识别标签。"""
        canvas = img_bgr.copy()
        x, y, w, h = (roi or self.roi)
        cv2.rectangle(canvas, (x, y), (x + w, y + h), (255, 200, 0), 2)
        canvas = put_text_cn(canvas, '识别区 ROI', 'ROI', (x + 6, y + 6), (255, 200, 0), 22)
        for color, blocks in det.items():
            for b in blocks:
                bx, by, bw, bh = b['rect']
                cv2.rectangle(canvas, (bx, by), (bx + bw, by + bh), COLOR_BGR[color], 2)
                canvas = put_text_cn(canvas, COLOR_CN[color], color, (bx, max(0, by - 26)),
                                     COLOR_BGR[color], 22)
        if label:
            canvas = put_text_cn(canvas, '识别结果：' + COLOR_CN.get(label, label),
                                 'Result: ' + label, (x, max(0, y - 34)),
                                 COLOR_BGR.get(label, (255, 255, 255)), 26)
        return canvas


reco = ColorRecognizer()

# 先用合成画面验证一次
color, block, det = reco.classify(DEMO_FRAME)
print('识别类别（类别字符串）=', repr(color), '=', COLOR_CN.get(color))
print('面积最大的色块 =', block)
show(reco.annotate(DEMO_FRAME, det, label=color))

In [ ]:
# ---------- §3 实时识别：ROI 框 + 色块框 + 类别标签，全部实时叠加 ----------
# 重复运行本 cell 会自动停掉上一个循环（不会两个线程同时抢摄像头）
t_live = start_live(recognizer=reco, annotate=True)

In [ ]:
# 停止实时循环（也可以直接点画面上的「停止实时」按钮）
stop_live()

In [ ]:
# ---------- 单帧识别（不依赖 ipywidgets 的实时循环，验收演示用这个也行） ----------
ok, frame = camera.read()
if ok and frame is not None:
    frame = cv2.resize(frame, (640, 480))
    label, block, det = reco.classify(frame)
    print('识别类别 = %s（%s）  类别字符串 = %r' % (COLOR_CN.get(label, label), label, label))
    print('最大色块 =', block)
    show(reco.annotate(frame, det, label=label))
else:
    print('读不到画面')

---
## §4　第 4 学时：HSV 标定与参数表

「标定」= 把**你现场这套灯光/相机/方块**实际对应的 HSV 上下限**测出来并记录下来**，而不是套用别人的数字。
下面给了三种办法，任选一种或组合使用。

| 方法 | 说明 | 适合 |
| --- | --- | --- |
| §4.1 滑条标定 | 拖动 H/S/V 滑条，右边实时看 Mask，调好点「应用到参数表」 | 最直观，推荐课堂用 |
| §4.2 ROI 采样 | 把方块放进 ROI，点一下按钮记录当前 HSV 均值 | 快速得到中心值 |
| §4.3 样本统计 | 用 `data/roi_image` 真实样本自动统计中位数 ± 容差 | 离线批量标定 |

In [ ]:
# ---------- §4.1 交互式 HSV 滑条标定（实时摄像头画面） ----------
def hsv_slider_tool(color='red', cam=None, width=640, height=480, fps=15):
    """拖动滑条，实时看摄像头画面的 Mask。调好后点「应用到参数表」。"""
    if not HAS_WIDGETS:
        print('没有 ipywidgets，跳过交互工具；可直接改 HSV_PARAMS 字典。')
        return None

    cam = cam or globals().get('camera')
    if cam is None:
        print('还没有相机对象，请先运行 §1 打开相机的 cell。')
        return None

    # 如果 §1.2/§3 的实时循环还在跑，先停掉，避免两个线程抢同一个摄像头
    try:
        stop_live(quiet=True)
    except Exception:
        pass

    lo, up = HSV_PARAMS[color]

    def sld(desc, mx, val):
        return widgets.IntSlider(value=int(val), min=0, max=mx, step=1, description=desc,
                                 continuous_update=True, layout=widgets.Layout(width='300px'))

    hm, sm, vm = sld('H_min', 180, lo[0]), sld('S_min', 255, lo[1]), sld('V_min', 255, lo[2])
    hM, sM, vM = sld('H_max', 180, up[0]), sld('S_max', 255, up[1]), sld('V_max', 255, up[2])
    img_w = widgets.Image(format='jpeg', width=400)
    mask_w = widgets.Image(format='jpeg', width=400)
    info = widgets.HTML()
    btn = widgets.Button(description='应用到参数表', button_style='primary')
    btn_stop = widgets.Button(description='停止实时', button_style='danger')
    display(widgets.VBox([widgets.HBox([img_w, mask_w]),
                          widgets.HBox([hm, hM]), widgets.HBox([sm, sM]), widgets.HBox([vm, vM]),
                          widgets.HBox([btn, btn_stop]), info]))

    state = {'stop': False, 'frame': None, 'n': 0}

    def cur():
        return (int(hm.value), int(sm.value), int(vm.value)), (int(hM.value), int(sM.value), int(vM.value))

    def apply(_b):
        HSV_PARAMS[color] = cur()
        try:
            reco.params = HSV_PARAMS
        except Exception:
            pass
        print('%s 阈值已更新为 lower=%s upper=%s' % (COLOR_CN[color], HSV_PARAMS[color][0], HSV_PARAMS[color][1]))

    def stop(_b):
        state['stop'] = True
        btn_stop.description = '已停止'
        btn_stop.disabled = True

    btn.on_click(apply)
    btn_stop.on_click(stop)

    # ★ 关键：取帧 + 转HSV + 生成Mask 全放在循环里，才会实时
    def loop():
        while not state['stop']:
            try:
                ok, f = cam.read()
            except Exception as e:
                print('读取画面失败:', repr(e)[:60])
                time.sleep(0.2)
                continue
            if not ok or f is None:
                time.sleep(0.05)
                continue

            f = cv2.resize(f, (width, height))
            state['frame'] = f
            state['n'] += 1
            lower, upper = cur()                      # 每帧读当前滑条值
            hsv = to_hsv(f)                           # 每帧重算
            mask = build_mask(hsv, color, {color: (lower, upper)})
            vis = f.copy()
            cv2.drawContours(vis, cv2.findContours(mask, cv2.RETR_EXTERNAL,
                                                   cv2.CHAIN_APPROX_SIMPLE)[-2],
                             -1, COLOR_BGR[color], 3)
            img_w.value = bgr8_to_jpeg(vis)
            mask_w.value = bgr8_to_jpeg(cv2.cvtColor(mask, cv2.COLOR_GRAY2BGR))
            if state['n'] % 5 == 0:
                info.value = ('<code>%s</code> 当前：lower = %s，upper = %s，Mask 白色占比 %.2f%%'
                              % (COLOR_CN[color], list(lower), list(upper),
                                 float((mask > 0).mean()) * 100))
            time.sleep(1.0 / fps)

    th = threading.Thread(target=loop, daemon=True)
    th.start()
    if '_LIVE' in globals():          # 登记给实时循环管理器，后面 start_live 会自动停掉它
        globals()['_LIVE']['thread'] = th
        globals()['_LIVE']['state'] = state
    print('实时标定已启动（摄像头画面）。调好点「应用到参数表」，结束点「停止实时」。')
    return dict(sliders=(hm, sm, vm, hM, sM, vM), button=btn, thread=th, state=state)


# 换颜色就改这里：color='red' / 'yellow' / 'green' / 'blue'
tool = hsv_slider_tool(color='yellow')

In [ ]:
# ---------- §4.2 在 ROI 内采样真实 HSV 均值 ----------
HSV_MEASURE = []      # 现场测得的数据，最后填进报告表


def measure_roi(img_bgr, color_truth=None, roi=None):
    """把方块放进 ROI，测出该区域的 HSV 均值并记录。"""
    roi = roi or ROI
    val = mean_hsv(img_bgr, roi)
    if val is None:
        print('ROI 为空'); return None
    row = {'颜色': COLOR_CN.get(color_truth, '?'), 'H均值': round(val[0], 1),
           'S均值': round(val[1], 1), 'V均值': round(val[2], 1)}
    HSV_MEASURE.append(row)
    print('采样 #%d：' % len(HSV_MEASURE), row)
    print('  -> 建议阈值  lower ≈ [%d, %d, %d]   upper ≈ [%d, %d, %d]'
          % (max(0, val[0] - 8), max(0, val[1] - 60), max(0, val[2] - 60),
             min(180, val[0] + 8), min(255, val[1] + 60), min(255, val[2] + 60)))
    return val


# 示例：把真实样本图当作“ROI 拍摄画面”来采样
for c in COLOR_ORDER:
    if c in SAMPLES:
        print('--- 真实样本 %s ---' % COLOR_CN[c])
        measure_roi(SAMPLES[c], color_truth=c, roi=(0, 0, SAMPLES[c].shape[1], SAMPLES[c].shape[0]))

In [ ]:
# ---------- §4.3 用真实样本自动统计阈值（中位数 ± 容差） ----------
def calibrate_from_samples(samples=None, h_pad=8, s_pad=60, v_pad=60):
    samples = SAMPLES if samples is None else samples
    params, rows = {}, []
    for color in COLOR_ORDER:
        if color not in samples:
            params[color] = HSV_PARAMS[color]
            continue
        pix = to_hsv(samples[color]).reshape(-1, 3).astype(np.float32)
        H, S, V = [float(t) for t in np.median(pix, axis=0)]
        lo = (int(max(0, H - h_pad)), int(max(0, S - s_pad)), int(max(0, V - v_pad)))
        up = (int(min(180, H + h_pad)), int(min(255, S + s_pad)), int(min(255, V + v_pad)))
        params[color] = (lo, up)
        rows.append({'颜色': COLOR_CN[color], 'H范围': '%d ~ %d' % (lo[0], up[0]),
                     'S范围': '%d ~ %d' % (lo[1], up[1]), 'V范围': '%d ~ %d' % (lo[2], up[2]),
                     '备注': '真实样本中位数±容差'})
    return params, rows


_, HSV_TABLE_STAT = calibrate_from_samples()
print('%-6s %-14s %-14s %-14s' % ('颜色', 'H范围', 'S范围', 'V范围'))
for r in HSV_TABLE_STAT:
    print('%-6s %-14s %-14s %-14s' % (r['颜色'], r['H范围'], r['S范围'], r['V范围']))
print('\n这张表可以直接抄进报告“5.1 HSV参数记录表”。是否采用由你决定（采样图可能偏色）。')

In [ ]:
# ---------- §4.4 保存参数表：hsv_config.json + hsv_params_table.csv ----------
# 把红色补齐成“两段”写法，方便其它程序（如 color_recognition.py）直接复用
HSV_PARAMS = {k: (tuple(int(t) for t in v[0]), tuple(int(t) for t in v[1])) for k, v in HSV_PARAMS.items()}


def hsv_table_rows(params):
    rows = []
    for c in COLOR_ORDER:
        lo, up = params[c]
        rows.append({'颜色': COLOR_CN[c], 'H范围': '%d-%d' % (lo[0], up[0]),
                     'S范围': '%d-%d' % (lo[1], up[1]), 'V范围': '%d-%d' % (lo[2], up[2]),
                     '测试效果': '待测', '备注': ''})
    return rows


def save_hsv_params(params, json_path=None, csv_path=None):
    """默认写到 OUTDIR（即 notebook 所在目录），不写死路径。"""
    json_path = json_path or os.path.join(OUTDIR, 'hsv_config.json')
    csv_path = csv_path or os.path.join(OUTDIR, 'hsv_params_table.csv')
    data = {c: [list(params[c][0]), list(params[c][1])] for c in COLOR_ORDER}
    with open(json_path, 'w', encoding='utf-8') as f:
        json.dump(data, f, ensure_ascii=False, indent=2)
    with open(csv_path, 'w', encoding='utf-8-sig', newline='') as f:
        w = csv.writer(f)
        w.writerow(['颜色', 'H_min', 'S_min', 'V_min', 'H_max', 'S_max', 'V_max'])
        for c in COLOR_ORDER:
            lo, up = params[c]
            w.writerow([COLOR_CN[c], lo[0], lo[1], lo[2], up[0], up[1], up[2]])
    print('已保存：', json_path, '、', csv_path)


save_hsv_params(HSV_PARAMS)
reco.params = HSV_PARAMS
print('\n当前使用的阈值：')
for c in COLOR_ORDER:
    print('  %-4s lower=%s upper=%s' % (COLOR_CN[c], HSV_PARAMS[c][0], HSV_PARAMS[c][1]))

---
## §5　40 次识别测试记录（正确率 ≥ 90%）

两种记录方式，任选：

- **方式一（现场，推荐验收用）**：把方块放进 ROI，看识别结果，点对应的「真实颜色」按钮记录一次。
- **方式二（一键生成）**：10 种光照条件 × 4 种颜色自动跑 40 组，生成可复现的 `test_record.csv`。

In [ ]:
# ---------- §5 公共记录工具 ----------
records = []
CSV_PATH = os.path.join(OUTDIR, 'test_record.csv')


def record_test(truth, pred, source='摄像头'):
    records.append({'序号': len(records) + 1, '时间': datetime.now().strftime('%H:%M:%S'),
                    '真实颜色': COLOR_CN.get(truth, truth), '识别颜色': COLOR_CN.get(pred, pred),
                    '是否正确': '正确' if truth == pred else '错误', '来源': source})
    return records[-1]


def summarize(rows):
    n = len(rows)
    c = sum(1 for r in rows if r.get('是否正确') == '正确')
    return n, c, (c / n * 100.0 if n else 0.0)


def save_records(rows=None, path=None):
    rows = records if rows is None else rows
    path = path or CSV_PATH
    if not rows:
        print('还没有记录'); return
    with open(path, 'w', encoding='utf-8-sig', newline='') as f:
        w = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
        w.writeheader(); w.writerows(rows)
    print('已保存 %d 条记录到 %s' % (len(rows), path))

In [ ]:
# ---------- §5 方式一：现场 40 次测试记录按钮 ----------
def record_tool(recognizer=None, cam=None):
    recognizer = recognizer or reco
    cam = cam or camera
    if not HAS_WIDGETS:
        print('没有 ipywidgets，请用 §5 方式二的一键生成。'); return None
    img_w = widgets.Image(format='jpeg', width=420)
    stat = widgets.HTML()
    out = widgets.Output()
    btns = {c: widgets.Button(description='真实=' + COLOR_CN[c], button_style='success') for c in COLOR_ORDER}
    btn_undo = widgets.Button(description='撤销一次')
    btn_save = widgets.Button(description='保存 CSV', button_style='primary')
    display(widgets.VBox([img_w, widgets.HBox(list(btns.values()) + [btn_undo, btn_save]), stat, out]))
    latest = {'color': 'none'}

    def refresh(*_):
        ok, f = cam.read()
        if ok and f is not None:
            f = cv2.resize(f, (640, 480))
            label, _b, det = recognizer.classify(f)
            latest['color'] = label
            img_w.value = bgr8_to_jpeg(recognizer.annotate(f, det, label=label))

    def upd():
        n, c, rate = summarize(records)
        stat.value = '<b>已记录 %d / 40　正确 %d　正确率 %.1f%%</b>' % (n, c, rate)

    def on_truth(color):
        r = record_test(color, latest['color'])
        with out:
            print('#%02d 真实=%-4s 识别=%-4s -> %s' % (r['序号'], r['真实颜色'], r['识别颜色'], r['是否正确']))
        upd()
        if len(records) >= 40:
            save_records()

    for c in COLOR_ORDER:
        btns[c].on_click(lambda b, cc=c: on_truth(cc))
    btn_undo.on_click(lambda b: (records.pop() if records else None, upd()))
    btn_save.on_click(lambda b: save_records())
    refresh(); upd()
    print('操作：把方块放进 ROI -> 运行本单元格刷新画面 -> 点对应“真实=xx”按钮记录一次。')
    return refresh


# refresh_once = record_tool()      # 取消注释即可现场记录

In [ ]:
# ---------- §5 方式二：一键生成 40 次测试记录（10 种光照 × 4 种颜色） ----------
def _synth_case(color, brightness=1.0, noise=6, hue_shift=0, cast=None):
    """合成一张“只有一种颜色方块”的测试图，带亮度/噪声/色偏干扰。"""
    img = make_demo_frame([(color, (215, 150, 140, 130))], jitter=0)
    if hue_shift:
        hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
        m = hsv[:, :, 1] > 10
        hh = hsv[:, :, 0].astype(np.int16)
        hh[m] = (hh[m] + hue_shift) % 180
        hsv[:, :, 0] = hh.astype(np.uint8)
        img = cv2.cvtColor(hsv, cv2.COLOR_HSV2BGR)
    f = img.astype(np.float32)
    if cast == 'warm':      # 暖光：红多蓝少
        f[:, :, 0] *= 0.86; f[:, :, 2] *= 1.12
    elif cast == 'cool':    # 冷光：蓝多红少
        f[:, :, 0] *= 1.15; f[:, :, 2] *= 0.88
    img = np.clip(f * brightness, 0, 255).astype(np.uint8)
    if noise:
        img = np.clip(img.astype(np.int16) + np.random.randint(-noise, noise + 1, img.shape), 0, 255).astype(np.uint8)
    return img


CONDITIONS = [
    ('正常光',   dict(brightness=1.00, noise=4)),
    ('正常光',   dict(brightness=0.95, noise=6, hue_shift=2)),
    ('正常光',   dict(brightness=0.90, noise=6, hue_shift=-2)),
    ('偏暗',     dict(brightness=0.80, noise=8)),
    ('偏暗',     dict(brightness=0.70, noise=8, hue_shift=2)),
    ('暖光色偏', dict(brightness=0.80, noise=8, cast='warm')),
    ('冷光色偏', dict(brightness=0.80, noise=8, cast='cool')),
    ('较暗',     dict(brightness=0.55, noise=10)),
    ('很暗',     dict(brightness=0.35, noise=12)),
    ('极限暗光', dict(brightness=0.26, noise=15)),
]


def benchmark_records(recognizer, roi=ROI, seed=2024):
    np.random.seed(seed)
    rows, idx = [], 0
    for cond, kw in CONDITIONS:
        for color in COLOR_ORDER:
            idx += 1
            pred, _b, _d = recognizer.classify(_synth_case(color, **kw), roi=roi)
            rows.append({'序号': idx, '时间': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
                         '真实颜色': COLOR_CN[color], '识别颜色': COLOR_CN.get(pred, '未识别'),
                         '是否正确': '正确' if pred == color else '错误',
                         '来源': '合成-亮度%.2f-%s' % (kw['brightness'], cond)})
    return rows


bench = benchmark_records(reco)
n, c, rate = summarize(bench)
save_records(bench)
print('自动测试：共 %d 次，正确 %d 次，正确率 %.1f%%' % (n, c, rate))
print('\n误识别用例（报告里“问题与分析”写这个）：')
bad = [r for r in bench if r['是否正确'] == '错误']
if bad:
    for r in bad:
        print('  #%02d 真实=%-4s 识别=%-4s (%s)' % (r['序号'], r['真实颜色'], r['识别颜色'], r['来源']))
else:
    print('  无（全部识别正确）')

In [ ]:
# ---------- §5 汇总已保存的记录（分颜色统计） ----------
def load_records(path):
    if not os.path.exists(path):
        return []
    with open(path, encoding='utf-8-sig', newline='') as f:
        return list(csv.DictReader(f))


rows_saved = load_records(CSV_PATH)
n, c, rate = summarize(rows_saved)
print('test_record.csv：共 %d 次，正确 %d 次，正确率 %.1f%%' % (n, c, rate))
per = {}
for r in rows_saved:
    k = r.get('真实颜色')
    per.setdefault(k, [0, 0])
    per[k][1] += 1
    if r.get('是否正确') == '正确':
        per[k][0] += 1
for k, (ok, tot) in per.items():
    print('   %-4s: %d/%d' % (k, ok, tot))
print('\n验收建议：40 次里正确 ≥ 36 次（≥90%），并且能说清至少一个误识别原因。')

---
## §6　课后作业：正常光 vs 较暗，以及阈值调整策略

比较两种光照下的识别结果，说明阈值怎么调。

In [ ]:
# ---------- §6 正常光 / 较暗对比 + V_min 自适应 ----------
def brightness_variant(img_bgr, factor):
    return np.clip(img_bgr.astype(np.float32) * factor, 0, 255).astype(np.uint8)


def test_brightness(recognizer, color='red', factors=(1.0, 0.8, 0.6, 0.4, 0.3, 0.25)):
    rows = []
    base = make_demo_frame([(color, (215, 150, 140, 130))])
    for f in factors:
        img = brightness_variant(base, f)
        pred, blk, _d = recognizer.classify(img)
        rows.append((f, pred, (blk['area'] if blk else 0)))
    return rows


print('%-8s %-8s %-8s' % ('亮度系数', '识别结果', '色块面积'))
for f, pred, area in test_brightness(reco, color='red'):
    print('%-10.2f %-10s %-8d' % (f, COLOR_CN.get(pred, pred), area))

print('\n结论：亮度降到某一档后 V 低于阈值的 V_min，Mask 直接变空 -> 漏检。')
print('调整策略：① 适当降低 V_min（如 50 -> 30）；② 固定相机曝光/白平衡，别用自动曝光；')
print('          ③ 光照对比强时先做直方图均衡化或加补光灯；④ 关键场合按光照分档设两组阈值。')

In [ ]:
# ---------- §6 自适应 V_min 演示：暗光下自动放宽 V_min ----------
def classify_adaptive(recognizer, img_bgr, roi=None):
    """先估一下画面亮度，再决定 V_min 要不要压低。"""
    v_mean = float(to_hsv(img_bgr).reshape(-1, 3)[:, 2].mean())
    params = {k: (list(v[0]), list(v[1])) for k, v in recognizer.params.items()}
    if v_mean < 120:                       # 画面偏暗 -> 放宽下限
        for k in params:
            params[k][0][2] = max(15, params[k][0][2] - 25)
    tmp = ColorRecognizer({k: (tuple(a), tuple(b)) for k, (a, b) in params.items()}, recognizer.roi)
    return v_mean, tmp.classify(img_bgr, roi)


print('%-10s %-8s %-10s %-10s %-10s' % ('亮度', 'V均值', '固定阈值', '自适应', '是否救回'))
base = make_demo_frame([('red', (215, 150, 140, 130))])
for f in (1.0, 0.6, 0.4, 0.3):
    img = brightness_variant(base, f)
    p1 = reco.classify(img)[0]
    v_mean, (p2, _b, _d) = classify_adaptive(reco, img)
    print('%-10.2f %-8.0f %-10s %-10s %-10s'
          % (f, v_mean, COLOR_CN.get(p1, p1), COLOR_CN.get(p2, p2), '是' if (p1 != 'red' and p2 == 'red') else '-'))

---
## §7*　拓展（加分）：手势识别 + 触发机械臂动作

报告“5.4 拓展实验”要求：**至少识别 3 种手势，并触发相应的机械臂动作**。

思路：先做**肤色分割 + 最大轮廓 + 凸包/凸缺陷**数伸出的手指根数，再映射成手势和动作。
（想升级成 PyTorch 版，可以把手势 ROI 存成图片训练一个小 CNN，接口保持 `predict_gesture(frame) -> 名称` 不变即可。）

In [ ]:
# ---------- §7.1 手势识别（OpenCV 版，数手指根数） ----------
GESTURES = {0: ('fist', '握拳'), 1: ('one', '1 指'), 2: ('two', '2 指'), 5: ('palm', '手掌')}


def skin_mask(img_bgr):
    """肤色分割（YCrCb 空间比 RGB 稳一些）。"""
    ycrcb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2YCrCb)
    m = cv2.inRange(ycrcb, np.array([0, 133, 77], np.uint8), np.array([255, 173, 127], np.uint8))
    m = cv2.morphologyEx(m, cv2.MORPH_OPEN, np.ones((5, 5), np.uint8))
    return cv2.morphologyEx(m, cv2.MORPH_CLOSE, np.ones((9, 9), np.uint8))


def count_fingers(mask):
    """用凸缺陷数手指（手指是掌心凸包和轮廓之间的缺口）。"""
    cnts = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)[-2]
    if not cnts:
        return 0, None
    cnt = max(cnts, key=cv2.contourArea)
    if cv2.contourArea(cnt) < 3000:
        return 0, cnt
    hull = cv2.convexHull(cnt, returnPoints=False)
    if hull is None or len(hull) < 4:
        return 0, cnt
    try:
        defects = cv2.convexityDefects(cnt, hull)
    except Exception:
        return 0, cnt
    fingers = 0
    if defects is not None:
        for i in range(defects.shape[0]):
            _s, _e, far, _d = defects[i, 0]
            p = tuple(cnt[far][0])
            a = tuple(cnt[defects[i, 0][0]][0])
            b = tuple(cnt[defects[i, 0][1]][0])
            la = np.linalg.norm(np.subtract(p, a)); lb = np.linalg.norm(np.subtract(p, b))
            if la * lb == 0:
                continue
            ang = np.degrees(np.arccos(np.clip(np.dot(np.subtract(a, p), np.subtract(b, p)) / (la * lb), -1, 1)))
            if ang < 90 and far > 0:
                fingers += 1
    return fingers + 1, cnt


def predict_gesture(img_bgr, roi=None):
    """返回 (手势名, 中文名, 画面)。没有手时返回 ('none', '未识别', ...)。"""
    vis = img_bgr.copy()
    x, y, w, h = roi if roi else (0, 0, img_bgr.shape[1], img_bgr.shape[0])
    sub = img_bgr[y:y + h, x:x + w]
    mask = skin_mask(sub)
    n, cnt = count_fingers(mask)
    if cnt is not None:
        cv2.drawContours(vis, [cnt + [x, y]], -1, (0, 255, 0), 2)
        hull = cv2.convexHull(cnt) + [x, y]
        cv2.drawContours(vis, [hull], -1, (255, 0, 0), 2)
    name, cn = GESTURES.get(n, ('none', '未识别'))
    vis = put_text_cn(vis, '手势：%s（%d 指）' % (cn, n), 'gesture: %s' % name,
                      (20, 30), (0, 255, 255), 26)
    return name, cn, vis


GESTURE_ACTIONS = {
    'fist': ('握拳', '夹爪闭合 gripper_close()'),
    'one':  ('1 指', '机械臂归位 home()'),
    'two':  ('2 指', '抬到拍摄位 set_tool_pose("capture_image")'),
    'palm': ('手掌', '夹爪张开 gripper_open()'),
}


def trigger_arm_action(gesture, arm=None):
    """把识别到的手势映射成机械臂动作；没有机械臂就只打印。"""
    cn, action = GESTURE_ACTIONS.get(gesture, ('未识别', '无动作'))
    print('手势 [%s] -> 机械臂动作：%s' % (cn, action))
    if arm is None:
        return
    try:
        if gesture == 'fist':
            arm.gripper_close()
        elif gesture == 'palm':
            arm.gripper_open()
        elif gesture == 'one':
            arm.home()
        elif gesture == 'two':
            arm.set_tool_pose(pose_name='capture_image')
    except Exception as e:
        print('机械臂动作失败：', repr(e)[:80])


# 用演示画面跑一遍流程（合成的“工作台”里没有手，所以会判为未识别，属正常）
for name in ['fist', 'one', 'two', 'palm']:
    trigger_arm_action(name)          # 没有机械臂 -> 只打印映射结果

In [ ]:
# ---------- §7.2 接上真实摄像头：手势 -> 机械臂动作 ----------
def get_arm(port=None):
    """尝试创建机械臂对象；没接硬件就返回 None（动作只打印）。"""
    try:
        from arm5dof_uservo import Arm5DoFUServo
        arm = Arm5DoFUServo() if port is None else Arm5DoFUServo(device=port)
        print('机械臂已连接')
        return arm
    except Exception as e:
        print('[INFO] 没有连接机械臂（%s），动作只做打印模拟。' % repr(e)[:60])
        return None


# arm = get_arm()                     # 有硬件时取消注释
# ok, f = camera.read(); f = cv2.resize(f, (640, 480))
# name, cn, vis = predict_gesture(f); show(vis); trigger_arm_action(name, arm=globals().get('arm'))
print('用法：拿掉摄像头前的遮挡，让手进入画面 -> predict_gesture(frame) -> trigger_arm_action(手势名)。')

---
## §8　实验结论与「误识别原因」分析

**1）完成的课堂任务**：固定 ROI 内识别红/黄/绿/蓝方块，画面显示类别并输出类别字符串 `red/yellow/green/blue`。

**2）提交产物**（本 notebook 运行完会自动生成）：

| 文件 | 内容 |
| --- | --- |
| `hsv_config.json` | 四色 HSV 阈值（JSON，供其它程序复用） |
| `hsv_params_table.csv` | 四色 HSV 参数表（填报告 5.1） |
| `test_record.csv` | 40 次识别测试记录与正确率（填报告 5.2） |

**3）常见误识别原因（验收时能说清一条即可）**

1. **环境光/白平衡漂移**：同一颜色在不同曝光、阴影下 H/S/V 都会变，固定阈值失效 → 重新标定或分档设阈值；
2. **反光高光**：色块表面反光处饱和度骤降、明度骤升，Mask 出现空洞 → 闭运算 / 降低 V_max；
3. **阴影**：阴影区 V 太低被漏掉 → 降低 V_min；
4. **阈值重叠**：黄色与绿色 H 区间相邻易混 → 收紧 H 区间；
5. **ROI 内多色块或背景干扰**：取最大轮廓时选错 → 缩小 ROI / 加强面积与矩形度过滤；
6. **运动模糊或目标太小**：轮廓破碎 → 稳一点、靠近一点。

**4）下一节预告**：把识别出的色块中心像素坐标经手眼标定换算成工作台坐标，驱动机械臂抓取分拣（见 `13.OpenCV 摄像头色块抓取分拣实验.ipynb`）。